## Load Model from DagsHub Registry
In this section I connect to the MLflow Model Registry on DagsHub to retrieve the "Champion" model. By using the latest version of the XGBoost model I ensure that the inference is performed using the most optimized and updated version from our experiments.

In [33]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os

# Setup MLflow and DagsHub connection
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML.mlflow")

# Load the best model from Registry
model_name = "XGBoost"
model_uri = f"models:/{model_name}/latest"
model = mlflow.sklearn.load_model(model_uri)
print(f"Model '{model_name}' loaded from cloud.")


Initialized MLflow to track repo "GigiSichinava/ML"

Repository GigiSichinava/ML initialized!

Model 'XGBoost' loaded from cloud.


## Test Data Preparation
To ensure the model functions correctly the test data must match the training data format. I load the Kaggle test set, identify the relevant numeric columns and handle missing values by filling them with zeros, mirroring the Cleaning steps from the training phase.

In [34]:
# Prepare Test Data
test_df = pd.read_csv('test.csv')

# Use same numeric columns as training and fill missing values
numeric_cols = test_df.select_dtypes(include=[np.number]).drop(['Id'], axis=1, errors='ignore').columns
X_test = test_df[numeric_cols].fillna(0)


## Final Prediction and Submission
Using the loaded model I generate price predictions for the test houses. The results are formatted into the required Kaggle structure and exported as submission.csv. This file represents the final output of the entire MLOps pipeline.

In [35]:
# Generate Predictions
predictions = model.predict(X_test)

# Create and save Kaggle submission
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": predictions
})

submission.to_csv('submission.csv', index=False)

print("Success: submission.csv generated.")

Success: submission.csv generated.
